In [8]:
!pip install -U nltk pymorphy2 pymorphy2-dicts-ru pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 308.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 6.6 MB/s eta 0:00:0000:0100:01


In [11]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [14]:
import re
import nltk
from nltk.corpus import stopwords
import pymorphy3
import pandas as pd

In [13]:
# Для русских текстов
morph = pymorphy3.MorphAnalyzer(lang='ru')
stop_words = set(stopwords.words('russian'))

def preprocess_text(text):
    # Удаление цензурных символов (****, ### и т.д.)
    text = re.sub(r'[*#]+', '', text)
    
    # Удаление спецсимволов, но сохранение основных знаков препинания
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s.,!?-]', ' ', text)
    
    # Приведение к нижнему регистру
    text = text.lower()
    
    # Токенизация
    tokens = text.split()
    
    # Удаление стоп-слов и лемматизация
    cleaned_tokens = []
    for token in tokens:
        if token not in stop_words and len(token) > 2:
            lemma = morph.parse(token)[0].normal_form
            cleaned_tokens.append(lemma)
    
    return ' '.join(cleaned_tokens)

In [52]:
table_data = pd.read_excel("../../data_raw/DataSet_V49 (2).xlsx")

In [53]:
text_data = pd.read_csv("../../anonymized_parsed_data_v8.csv")

In [54]:
text_data

,file_name,patient_id,anonymized_text_data
0,АсатрянРС_analyzes_0.html,7365.0,"Подождите, пожалуйста... Идет подготовка к выв..."
1,СтаценкоНА_analyzes_0.html,18865.0,"Подождите, пожалуйста... Идет подготовка к выв..."
2,ДмитриеваЛН_analyzes_0.html,3137.0,"Подождите, пожалуйста... Идет подготовка к выв..."
3,ПетровЕГ_analyzes_0.html,554.0,"Подождите, пожалуйста... Идет подготовка к выв..."
4,ВласенкоВГ_analyzes_0.html,1504.0,"Подождите, пожалуйста... Идет подготовка к выв..."
...,...,...,...
4398,ГорчаковаДМ_analyzes_0.html,12845.0,"Подождите, пожалуйста... Идет подготовка к выв..."
4399,КорниенкоЛД_analyzes_0.html,13202.0,"Подождите, пожалуйста... Идет подготовка к выв..."
4400,КоноваловВадимГерманович14917_analyzes_0.html,14917.0,"Подождите, пожалуйста... Идет подготовка к выв..."
4401,КоробейниковЕА_analyzes_0.html,19044.0,"Подождите, пожалуйста... Идет подготовка к выв..."


In [17]:
text_data["patient_id"] = text_data["patient_id"].astype(object)

In [18]:
text_data

,file_name,patient_id,anonymized_text_data
0,АсатрянРС_analyzes_0.html,7365.0,"Подождите, пожалуйста... Идет подготовка к выв..."
1,СтаценкоНА_analyzes_0.html,18865.0,"Подождите, пожалуйста... Идет подготовка к выв..."
2,ДмитриеваЛН_analyzes_0.html,3137.0,"Подождите, пожалуйста... Идет подготовка к выв..."
3,ПетровЕГ_analyzes_0.html,554.0,"Подождите, пожалуйста... Идет подготовка к выв..."
4,ВласенкоВГ_analyzes_0.html,1504.0,"Подождите, пожалуйста... Идет подготовка к выв..."
...,...,...,...
4398,ГорчаковаДМ_analyzes_0.html,12845.0,"Подождите, пожалуйста... Идет подготовка к выв..."
4399,КорниенкоЛД_analyzes_0.html,13202.0,"Подождите, пожалуйста... Идет подготовка к выв..."
4400,КоноваловВадимГерманович14917_analyzes_0.html,14917.0,"Подождите, пожалуйста... Идет подготовка к выв..."
4401,КоробейниковЕА_analyzes_0.html,19044.0,"Подождите, пожалуйста... Идет подготовка к выв..."


In [55]:
table_data.dropna(subset=['Смерть'], inplace=True)

In [21]:
df2_subset = table_data[['Код пациента', 'Смерть']]
df2_subset.rename(columns={'Код пациента': 'patient_id', 'Смерть': 'label'}, inplace=True)

df2_subset["patient_id"] = df2_subset["patient_id"].astype(str)
text_data['patient_id'] = text_data['patient_id'].apply(lambda x: str(int(x)) if pd.notna(x) else None)

# Делаем merge по идентификатору
df1 = text_data.merge(df2_subset, on='patient_id', how='inner')

/tmp/ipykernel_2110872/4045082474.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_subset.rename(columns={'Код пациента': 'patient_id', 'Смерть': 'label'}, inplace=True)
/tmp/ipykernel_2110872/4045082474.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_subset["patient_id"] = df2_subset["patient_id"].astype(str)


In [23]:
df1["label"].value_counts()

label
Нет    697
Да      33
Name: count, dtype: int64

In [59]:
df1['anonymized_text_data'] = df1['anonymized_text_data'].apply(lambda x: preprocess_text(x))

In [57]:
df2_subset = table_data[['Файл(Анализы)', 'Смерть']]
df2_subset.rename(columns={'Файл(Анализы)': 'file_name', 'Смерть': 'label'}, inplace=True)

df2_subset["file_name"] = df2_subset["file_name"].astype(str)
# text_data['file_name'] = text_data['file_name'].apply(lambda x: str(int(x)) if pd.notna(x) else None)

# Делаем merge по идентификатору
df1 = text_data.merge(df2_subset, on='file_name', how='inner')

/tmp/ipykernel_2110872/3919788308.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_subset.rename(columns={'Файл(Анализы)': 'file_name', 'Смерть': 'label'}, inplace=True)
/tmp/ipykernel_2110872/3919788308.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_subset["file_name"] = df2_subset["file_name"].astype(str)


In [58]:
df1

,file_name,patient_id,anonymized_text_data,label
0,АсатрянРС_analyzes_0.html,7365.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
1,ПетровЕГ_analyzes_0.html,554.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
2,ДиктяревАВ_analyzes_0.html,10765.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
3,СвергунСЛ3428_analyzes_0.html,3428.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
4,КаргинСП5884_analyzes_0.html,5884.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
...,...,...,...,...
2875,БелоцерковецЖП4941_analyzes_0.html,4941.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
2876,ЛегаевГФ_analyzes_0.html,10622.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
2877,ГорчаковаДМ_analyzes_0.html,12845.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет
2878,КорниенкоЛД_analyzes_0.html,13202.0,"Подождите, пожалуйста... Идет подготовка к выв...",Нет


In [60]:
df1['anonymized_text_data'][0]

'подождите, пожалуйста... идти подготовка вывод обследование воздействий. это занять несколько минут. 100 учреждение государственный бюджетный учреждение здравоохранение приморский краевой клинический больница общий анализ кровь экспресс выполнить 03.05.2019 ф.и.о. фио дата рождение 24.03.1953 год отделение реанимация интенсивный терапия рсц палата медицинский карта 7365 диагноз ибс. нестабильный стенокардия умеренный риска. пикс. осн killip фон хсн ст., ф.к. направить врач-реаниматолог фио дата направление 03.05.2019 дата взятие биоматериал 03.05.2019 время исследовать компонент результат референсный значение единица измерение лейкоцит wbc 6.7 3.9-9 лимфоцит lym 0.8 1.2-3 лимфоцит lym 11.9 19-37 эритроцит rbc 3.85 3.9-5 гемоглобин hgb 116 120-160 гематокрит hct 30.4 36-48 средний объём эритроцит mcv 79.1 80-100 средний содержание гемоглобин эритроцит mch 30.1 27-31 ср. концентрация гемоглобин эритроцит mchc 381 300-380 тромбоцит plt 145 150-400 средний объём тромбоцит mpv 7.9 7.4-10.4

In [30]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 9.6 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 8.6 MB/s eta 0:00:00


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec, FastText

In [61]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 3),
    analyzer='char_wb',
    sublinear_tf=True
)

X_texts = vectorizer.fit_transform(df1["anonymized_text_data"])

In [34]:
X_texts

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1212588 stored elements and shape (730, 10000)>

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

In [62]:
X_train, X_test, y_train, y_test = train_test_split(
    X_texts, df1["label"], test_size=0.2, random_state=42, stratify=df1["label"]
)


In [63]:
text_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

text_model.fit(X_train, y_train)

y_pred_proba = text_model.predict_proba(X_test)[:, 1]
y_pred = text_model.predict(X_test)
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC на тесте: {auc:.4f}")

AUC на тесте: 0.9897


In [65]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          Да       0.54      0.90      0.68        21
         Нет       1.00      0.97      0.98       555

    accuracy                           0.97       576
   macro avg       0.77      0.94      0.83       576
weighted avg       0.98      0.97      0.97       576

